# 01 - Data Exploration
# استكشاف البيانات

This notebook provides an introduction to the Elite Set-Piece Analytics platform
and explores the available data sources.

## Contents
1. Setup and Imports
2. Loading Data
3. Data Overview
4. Set-Piece Extraction
5. Basic Visualizations
6. Data Quality Checks

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add source directory to path
import sys
sys.path.insert(0, '../src')

# Project imports
from data.loaders import WyscoutLoader, StatsBombLoader, MetricaLoader, UnifiedDataLoader
from data.extractors import SetPieceExtractor
from visualization.pitch import PitchVisualizer
from utils.config import Config

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
%matplotlib inline

print("✅ Imports successful!")
print(f"Project version: 0.1.0")

## 2. Loading Data

We support multiple data sources:
- **StatsBomb**: High-quality event data with 360 tracking
- **Wyscout**: Comprehensive event data
- **Metrica**: Sample tracking data

In [ ]:
# Initialize data loaders
statsbomb_loader = StatsBombLoader(use_api=True)
unified_loader = UnifiedDataLoader()

print("📁 Data loaders initialized")

In [ ]:
# Load StatsBomb competitions (free tier)
try:
    competitions = statsbomb_loader.load_competitions()
    print(f"Available competitions: {len(competitions)}")
    display(competitions.head(10))
except Exception as e:
    print(f"Note: Could not load competitions - {e}")
    print("This is normal if you don't have internet access or statsbombpy installed.")

In [ ]:
# Try to load World Cup 2022 data (if available)
try:
    # World Cup 2022: competition_id=43, season_id=106
    matches = statsbomb_loader.load_matches(competition_id=43, season_id=106)
    print(f"World Cup 2022 matches: {len(matches)}")
    display(matches.head())
except Exception as e:
    print(f"Note: Could not load World Cup matches - {e}")
    print("Creating sample data for demonstration...")
    
    # Create sample data
    matches = pd.DataFrame({
        'match_id': [1, 2, 3, 4, 5],
        'home_team': ['Argentina', 'France', 'Morocco', 'Croatia', 'Brazil'],
        'away_team': ['France', 'Morocco', 'Croatia', 'Brazil', 'Netherlands'],
        'home_score': [3, 2, 0, 1, 1],
        'away_score': [3, 0, 2, 0, 1]
    })

## 3. Data Overview

Let's explore the structure of our event data.

In [ ]:
# Create sample event data for exploration
sample_events = pd.DataFrame({
    'event_id': range(1, 101),
    'event_type': np.random.choice(['Pass', 'Shot', 'Corner', 'Free Kick', 'Dribble'], 100),
    'x': np.random.uniform(0, 105, 100),
    'y': np.random.uniform(0, 68, 100),
    'timestamp': np.sort(np.random.uniform(0, 5400, 100)),
    'team': np.random.choice(['Home', 'Away'], 100),
    'player': [f'Player_{i}' for i in np.random.randint(1, 23, 100)]
})

print("Sample events shape:", sample_events.shape)
display(sample_events.head(10))

In [ ]:
# Event type distribution
event_counts = sample_events['event_type'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=event_counts.index, y=event_counts.values, palette='viridis')
plt.title('Event Type Distribution', fontsize=14)
plt.xlabel('Event Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Set-Piece Extraction

Extract and analyze set-piece events from the data.

In [ ]:
# Initialize extractor
extractor = SetPieceExtractor()

# Extract set-pieces
set_pieces = extractor.extract_all(sample_events)

print(f"Total set-pieces extracted: {len(set_pieces)}")
if len(set_pieces) > 0:
    print("\nSet-piece type distribution:")
    print(set_pieces['set_piece_type'].value_counts())

In [ ]:
# Calculate statistics
stats = extractor.calculate_set_piece_statistics(set_pieces)
print("Set-piece statistics:")
print(stats)

## 5. Basic Visualizations

Visualize events on a football pitch.

In [ ]:
# Initialize pitch visualizer
pitch_viz = PitchVisualizer()

# Draw empty pitch
fig, ax = pitch_viz.draw_pitch()
pitch_viz.add_title(ax, "Football Pitch", "Standard FIFA Dimensions")
plt.show()

In [ ]:
# Plot events on pitch
fig, ax = pitch_viz.draw_pitch()

# Plot corners
corners = sample_events[sample_events['event_type'] == 'Corner']
ax.scatter(corners['x'], corners['y'], c='yellow', s=100, 
           edgecolors='black', zorder=5, label='Corners')

# Plot shots
shots = sample_events[sample_events['event_type'] == 'Shot']
ax.scatter(shots['x'], shots['y'], c='red', s=100, 
           marker='*', zorder=5, label='Shots')

pitch_viz.add_title(ax, "Event Locations", "Corners and Shots")
ax.legend(loc='upper left')
plt.show()

## 6. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing values:")
print(sample_events.isnull().sum())

In [ ]:
# Check coordinate ranges
print("\nCoordinate ranges:")
print(f"X: {sample_events['x'].min():.2f} to {sample_events['x'].max():.2f} (expected: 0 to 105)")
print(f"Y: {sample_events['y'].min():.2f} to {sample_events['y'].max():.2f} (expected: 0 to 68)")

In [ ]:
# Summary statistics
print("\nSummary statistics:")
display(sample_events.describe())

## Next Steps

1. **02_set_piece_extraction.ipynb**: Deep dive into set-piece extraction
2. **03_feature_engineering.ipynb**: Create features for prediction
3. **04_model_training.ipynb**: Train the first receiver predictor
4. **05_tactical_analysis.ipynb**: Analyze tactical patterns
5. **06_visualization.ipynb**: Create advanced visualizations

---

**Note**: Make sure to run `scripts/download_data.sh` to download the latest data.